In [48]:
import torch
import torch.nn as nn
import torch.optim as opt
import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
nltk.download('punkt_tab')
nltk.download('wordnet')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [15]:
dataset = pd.read_csv("/content/sentiment_analysis.csv")
dataset.head(4)

,Year,Month,Day,Time of Tweet,text,sentiment,Platform
0,2018,8,18,morning,What a great day!!! Looks like dream.,positive,Twitter
1,2018,8,18,noon,"I feel sorry, I miss you here in the sea beach",positive,Facebook
2,2017,8,18,night,Don't angry me,negative,Facebook
3,2022,6,8,morning,We attend in the class just for listening teac...,negative,Facebook


In [16]:
dataset = dataset[["text", "sentiment"]]

In [17]:
dataset.head(3)

,text,sentiment
0,What a great day!!! Looks like dream.,positive
1,"I feel sorry, I miss you here in the sea beach",positive
2,Don't angry me,negative


In [18]:
dataset["sentiment"] = dataset["sentiment"].map({
    "positive" : 1,
    "negative" : -1,
    "neutral" : 0
})

In [19]:
dataset["sentiment"]

,sentiment
0,1
1,1
2,-1
3,-1
4,-1
...,...
494,-1
495,-1
496,0
497,1


In [20]:
# convert text into emdeding then, apply LSTM on it.

X = dataset["text"]
y = dataset["sentiment"]

In [21]:
stemitization =  PorterStemmer()

def clean(text):

  text =  re.sub(r'[^a-zA-Z0-9 ]', '', text)

  text = text.lower()

  tokens = text.split()

  tokens = [stemitization.stem(token) for token in tokens]


  return tokens

In [22]:
X = X.apply(clean)


In [36]:
# Hyperparameters

vocab_size =  5000
max_length = 100
embed_dim = 100
hidden_dim = 64

In [24]:
dataset["sentiment"].unique()

array([ 1, -1,  0])

In [25]:
# Calculating Vocab Size
all_token = [token for tokens in X for token in tokens]
counter = Counter(all_token).most_common(vocab_size - 2)
word2idx = {'<PAD>': 0, '<UNK>': 1}

for word, _ in counter:
   word2idx[word] = len(word2idx)

vocab_size = len(word2idx)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 1291


In [26]:
# Encoding
def Encode(token):
  ids = [word2idx.get(tok, 1) for tok in token]
  if len(ids) >= max_length:
    return ids[:max_length]
  else:
    return ids + [0] * (max_length - len(ids))

# Vectorization
X_encode = np.array([Encode(token) for token in X], dtype = np.int64)
y = y.values.astype(np.float32).reshape(-1,1)

In [29]:
# Train test split

X_train, X_test, y_train, y_test = train_test_split(X_encode, y, test_size = 0.2, random_state = 42)


In [53]:
X_train = torch.tensor(X_train, dtype = torch.long)
X_test = torch.tensor(X_test, dtype = torch.long)
y_train = torch.tensor(y_train, dtype = torch.float32)
y_test = torch.tensor(y_test, dtype = torch.float32)

/tmp/ipykernel_13858/316653357.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype = torch.long)
/tmp/ipykernel_13858/316653357.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test, dtype = torch.long)
/tmp/ipykernel_13858/316653357.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype = torch.float32)
/tmp/ipykernel_13858/316653357.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() 

In [54]:
#  LSTM model

class SentimentLSTM(nn.Module):

  def __init__ (self, vocab_size, embed_dim, hidden_dim):
    super().__init__()

    self.emdeding = nn.Embedding(num_embeddings = vocab_size, embedding_dim = embed_dim, padding_idx = 0)
    self.lstm = nn.LSTM(embed_dim, hidden_dim, dropout = 0.2, batch_first = True)
    self.fc = nn.Linear(hidden_dim, 1)

  def forward(self, x):

    x = self.emdeding(x)
    out, _ = self.lstm(x)
    out = out[:,-1,:]
    out = self.fc(out)

    return out

In [55]:
model = SentimentLSTM(vocab_size, embed_dim, hidden_dim)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [62]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = opt.Adam(model.parameters(), lr = 0.001)

In [63]:
# Training loop

for epoch in range(100):

  # model.train()

  output = model.forward(X_train)

  loss = loss_fn(output, y_train)

  # Back prop
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  model.eval()

  with torch.no_grad():
    test_out = model(X_test)
    pred =  (torch.sigmoid(test_out) > 0.5).float()
    acc = (pred == y_test).float().mean().item()


  print(f" {epoch + 1} and the loss is {loss}, and acc is  {acc}")

 1 and the loss is 0.7017106413841248, and acc is  0.3400000035762787
 2 and the loss is 0.6959289908409119, and acc is  0.30000001192092896
 3 and the loss is 0.6901974678039551, and acc is  0.30000001192092896
 4 and the loss is 0.6844958662986755, and acc is  0.30000001192092896
 5 and the loss is 0.6788042783737183, and acc is  0.30000001192092896
 6 and the loss is 0.673093855381012, and acc is  0.30000001192092896
 7 and the loss is 0.6673233509063721, and acc is  0.30000001192092896
 8 and the loss is 0.6614416837692261, and acc is  0.30000001192092896
 9 and the loss is 0.6553908586502075, and acc is  0.30000001192092896
 10 and the loss is 0.6491087079048157, and acc is  0.30000001192092896
 11 and the loss is 0.6425279378890991, and acc is  0.30000001192092896
 12 and the loss is 0.6355744004249573, and acc is  0.30000001192092896
 13 and the loss is 0.6281639337539673, and acc is  0.30000001192092896
 14 and the loss is 0.6201977133750916, and acc is  0.30000001192092896
 15